In [14]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.sparse import csr_matrix
from math import sqrt
import gc

# ===============================
# 1. VERİYİ YÜKLE VE TEMİZLE
# ===============================
path = "Users/seraykeskinkilinc/" 

books = pd.read_csv(f"{path}books.csv")
ratings = pd.read_csv(f"{path}ratings.csv")
book_tags = pd.read_csv(f"{path}book_tags.csv")
tags = pd.read_csv(f"{path}tags.csv")

# Kalite Filtresi
popular_books = ratings['book_id'].value_counts()
popular_books = popular_books[popular_books > 2000].index
ratings = ratings[ratings['book_id'].isin(popular_books)]

active_users = ratings['user_id'].value_counts()
active_users = active_users[active_users > 50].index
ratings = ratings[ratings['user_id'].isin(active_users)]

# Train/Test Split
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)
print(f"✅ Filtreleme sonrası: {len(train_data)} eğitim satırı ile başlanıyor.")

# ===============================
# 2. İÇERİK ANALİZİ (TF-IDF)
# ===============================
book_tags = book_tags.merge(tags, on='tag_id')
book_content = book_tags.groupby('goodreads_book_id')['tag_name'].apply(lambda x: " ".join(x.astype(str))).reset_index()
books_with_content = books.merge(book_content, on='goodreads_book_id', how='inner')

tfidf = TfidfVectorizer(stop_words='english', max_features=1000)
tfidf_matrix = tfidf.fit_transform(books_with_content['tag_name'])

# ===============================
# 3. COLLABORATIVE VE POPÜLERLİK SKORLARI
# ===============================
user_book_matrix = train_data.pivot_table(index='book_id', columns='user_id', values='rating').fillna(0)
user_book_sparse = csr_matrix(user_book_matrix.values)
collab_sim = cosine_similarity(user_book_sparse)
collab_sim_df = pd.DataFrame(collab_sim, index=user_book_matrix.index, columns=user_book_matrix.index)

# Popülerlik skoru (0-1 arası normalize)
book_counts = train_data['book_id'].value_counts()
popularity_scores = book_counts / book_counts.max()

# ===============================
# 4. TURBO HİBRİT HESAPLAMA MOTORU
# ===============================
def get_hybrid_scores_turbo(target_book_id, alpha=0.4):
    """
    Precision artırmak için yazar ve popülerlik bonusu eklenmiş motor.
    """
    idx_list = books_with_content[books_with_content['book_id'] == target_book_id]
    if idx_list.empty: return None
    
    target_idx = idx_list.index[0]
    target_author = books_with_content.iloc[target_idx]['authors']
    
    # a. İçerik Skoru
    content_scores = cosine_similarity(tfidf_matrix[target_idx], tfidf_matrix).flatten()
    
    # b. Collaborative Skoru
    collab_scores = np.zeros(len(content_scores))
    if target_book_id in collab_sim_df.index:
        c_raw = collab_sim_df[target_book_id]
        for bid, score in c_raw.items():
            m_idx = books_with_content[books_with_content['book_id'] == bid].index
            if not m_idx.empty:
                collab_scores[m_idx[0]] = score
                
    # c. Yazar Bonusu (%20)
    author_bonus = np.where(books_with_content['authors'] == target_author, 0.2, 0)
    
    # d. Popülerlik Bonusu (%10)
    pop_bonus = np.array([popularity_scores.get(bid, 0) * 0.1 for bid in books_with_content['book_id']])
    
    # Harmanlama
    return (content_scores * alpha) + (collab_scores * (1 - alpha)) + author_bonus + pop_bonus

def get_netflix_style_recs(title, n=5):
    idx_list = books_with_content[books_with_content['title'].str.contains(title, case=False, na=False)]
    if idx_list.empty: return print("Kitap bulunamadı!")
    
    target_book_id = idx_list.iloc[0]['book_id']
    final_scores = get_hybrid_scores_turbo(target_book_id)
    
    books_with_content['tmp_score'] = final_scores
    results = books_with_content[books_with_content['book_id'] != target_book_id]
    results = results.sort_values(by='tmp_score', ascending=False).head(n)
    
    print(f"\n'{title}' kitabını beğendiysen şunları da beğenebilirsin:")
    for i, row in results.iterrows():
        print(f"{i+1}. {row['title']} (Yazar: {row['authors']}) - Skor: %{row['tmp_score']*100:.1f}")

# ===============================
# 5. PERFORMANS TESTLERİ (RMSE & PRECISION)
# ===============================
def calculate_rmse(num_samples=2000):
    print(f"\nRMSE hesaplanıyor...")
    samples = test_data.sample(min(len(test_data), num_samples))
    y_true, y_pred = [], []
    global_mean = train_data['rating'].mean()

    for _, row in samples.iterrows():
        u_id, b_id, actual = row['user_id'], row['book_id'], row['rating']
        prediction = global_mean
        if b_id in collab_sim_df.index:
            sim_books = collab_sim_df[b_id].sort_values(ascending=False)[1:11]
            user_ratings = train_data[train_data['user_id'] == u_id]
            w_sum, s_sum = 0, 0
            for s_bid, s_score in sim_books.items():
                match = user_ratings[user_ratings['book_id'] == s_bid]
                if not match.empty:
                    w_sum += match.iloc[0]['rating'] * s_score
                    s_sum += s_score
            if s_sum > 0: prediction = w_sum / s_sum
        y_true.append(actual)
        y_pred.append(prediction)
    
    score = sqrt(mean_squared_error(y_true, y_pred))
    print(f"✅ Model RMSE Skoru: {score:.4f}")

def calculate_advanced_metrics(all_recommendations, tfidf_matrix, books_with_content):
    """
    Kapsam (Coverage) ve Çeşitlilik (Diversity) hesaplayan fonksiyon.
    """
    # 1. KATALOG KAPSAMI (COVERAGE)
    # Tüm test kullanıcılarına önerilen toplam benzersiz kitap sayısı
    unique_recommended_books = set([item for sublist in all_recommendations for item in sublist])
    total_books_in_catalog = len(books_with_content)
    coverage = (len(unique_recommended_books) / total_books_in_catalog) * 100

    # 2. LİSTE İÇİ ÇEŞİTLİLİK (INTRA-LIST DIVERSITY)
    # Önerilen kitapların birbirine ne kadar benzemediği (1 - Cosine Similarity)
    diversity_scores = []
    
    for rec_ids in all_recommendations:
        if len(rec_ids) < 2: continue
        
        # Önerilen kitapların TF-IDF matrisindeki indekslerini bul
        indices = books_with_content[books_with_content['book_id'].isin(rec_ids)].index.tolist()
        if len(indices) < 2: continue
        
        # Bu kitaplar arasındaki benzerlik matrisini hesapla
        sim_matrix = cosine_similarity(tfidf_matrix[indices])
        
        # Üst üçgeni al (kendisiyle olan benzerlikleri hariç tutmak için)
        upper_indices = np.triu_indices(len(indices), k=1)
        avg_similarity = np.mean(sim_matrix[upper_indices])
        
        # Çeşitlilik = 1 - Ortalama Benzerlik
        diversity_scores.append(1 - avg_similarity)

    avg_diversity = np.mean(diversity_scores) if diversity_scores else 0
    
    return coverage, avg_diversity

# Güncellenmiş Test Fonksiyonu
def evaluate_advanced(k=5, num_users=200):
    print(f"\nKapsamlı Model Analizi Başlatılıyor ({num_users} kullanıcı)...")
    
    user_counts = test_data[test_data['rating'] >= 4]['user_id'].value_counts()
    qualified_users = user_counts[user_counts >= 5].index
    test_users = np.random.choice(qualified_users, min(num_users, len(qualified_users)), replace=False)
    
    precision_scores = []
    all_recommended_ids = []

    for user in test_users:
        actual_relevant = test_data[(test_data['user_id'] == user) & (test_data['rating'] >= 4)]['book_id'].values
        user_favs = train_data[train_data['user_id'] == user].sort_values(by='rating', ascending=False)
        seed_id = user_favs.iloc[0]['book_id']
        
        scores = get_hybrid_scores_turbo(seed_id)
        if scores is not None:
            top_k_idx = np.argsort(scores)[-(k+1):-1][::-1]
            rec_ids = books_with_content.iloc[top_k_idx]['book_id'].values
            
            # Hassasiyet Skoru
            hits = len(set(rec_ids) & set(actual_relevant))
            precision_scores.append(hits / k)
            
            # Kapsam ve Çeşitlilik için önerileri sakla
            all_recommended_ids.append(rec_ids.tolist())

    # Metrikleri Hesapla
    coverage, diversity = calculate_advanced_metrics(all_recommended_ids, tfidf_matrix, books_with_content)
    
    print(f"\n--- FİNAL RAPORU ---")
    print(f"Precision@5: %{np.mean(precision_scores)*100:.2f}")
    print(f"Katalog Kapsamı (Coverage): %{coverage:.2f}")
    print(f"Çeşitlilik Skoru (Diversity): {diversity:.4f}")

# Çalıştır
evaluate_advanced(num_users=1000)

# ===============================
# FİNAL ÇALIŞTIRMA
# ===============================
calculate_rmse()
get_netflix_style_recs("Dune")

StatementMeta(b2d3938f-b61c-4394-b3df-be7add6868e1, 5, 19, Finished, Available, Finished)

✅ Filtreleme sonrası: 1183774 eğitim satırı ile başlanıyor.

Kapsamlı Model Analizi Başlatılıyor (1000 kullanıcı)...

--- FİNAL RAPORU ---
Precision@5: %9.28
Katalog Kapsamı (Coverage): %6.69
Çeşitlilik Skoru (Diversity): 0.2564

RMSE hesaplanıyor...
✅ Model RMSE Skoru: 0.9640

'Dune' kitabını beğendiysen şunları da beğenebilirsin:
70. Ender's Game (Ender's Saga, #1) (Yazar: Orson Scott Card) - Skor: %55.0
276. Foundation (Foundation #1) (Yazar: Isaac Asimov) - Skor: %51.7
2491. Heretics of Dune (Dune Chronicles #5) (Yazar: Frank Herbert) - Skor: %51.0
54. The Hitchhiker's Guide to the Galaxy (Hitchhiker's Guide to the Galaxy, #1) (Yazar: Douglas Adams) - Skor: %50.8
2045. God Emperor of Dune (Dune Chronicles #4) (Yazar: Frank Herbert) - Skor: %50.4
